## Model

In [6]:
import numpy as np

class CustomLinearSVM:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):

        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters        
        self.w = None
        self.b = None

    def fit(self, X, y):
        
        n_samples, n_features = X.shape
        
        self.w = np.zeros(n_features)
        self.b = 0.0
        
        y_mirrored = np.where(y <= 0, -1, 1)

        for epoch in range(self.n_iters):
            
            for idx, x_i in enumerate(X):
                
                y_i = y_mirrored[idx]
                f_i= np.dot((self.w).T, x_i)+self.b

                if f_i * y_i <1:
                    self.w=self.w-self.lr*(self.lambda_param*self.w - y_i*x_i)
                    self.b = self.b + self.lr * y_i
                
                else:
                    self.w=self.w-self.lr*self.lambda_param*self.w 

                
              

    def predict(self, X):
       linear_output = np.dot(X, self.w) + self.b
       prediction = np.where(linear_output >= 0, 1, 0)
       return prediction
       


class OneVsRestSVM:
    def __init__(self, n_classes=10, learning_rate=0.001, lambda_param=0.01, n_iters=50):
        self.n_classes = n_classes
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.models = []

    def fit(self, X, y):
        for i in range(self.n_classes):
            print(f"    Training binary model for Class {i} vs Rest...")
            
            y_binary = np.where(y == i, 1, 0)
            
            binary_model = CustomLinearSVM(
                learning_rate=self.lr, 
                lambda_param=self.lambda_param, 
                n_iters=self.n_iters
            )
            binary_model.fit(X, y_binary)
            self.models.append(binary_model)

    def predict(self, X):
        n_samples = X.shape[0]
        scores = np.zeros((n_samples, self.n_classes))
        
        for i, model in enumerate(self.models):
            raw_value = np.dot(X, model.w) + model.b
            scores[:, i] = raw_value
        
        winning_classes = np.argmax(scores, axis=1)
        return winning_classes

## Training
#### HOG

In [2]:
import time

from preprocessing2 import preprocess
from sklearn.metrics import classification_report, confusion_matrix

X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="hog", n_pca=50)

# ==========================================
# 4. TRAIN AND EVALUATE
# ==========================================
print("\nInitializing Custom 10-Class SVM (OvR)...")
ovr_svm = OneVsRestSVM(n_classes=10, learning_rate=0.001, lambda_param=0.01, n_iters=50)

print("Training all 10 models... (This might take a few minutes in Python)")
start_time = time.time()
ovr_svm.fit(X_train, y_train)
print(f"Full 10-Class Training completed in {(time.time() - start_time):.2f} seconds.")

print("\nEvaluating on Validation Data...")
val_preds = ovr_svm.predict(X_val)

print("\n" + "="*50)
print("  10-CLASS MODEL PERFORMANCE (HOG + PCA FEATURES)")
print("="*50)
target_names = [f"Digit {i}" for i in range(10)]
print(classification_report(y_val, val_preds, target_names=target_names))


print("\nValidation Confusion Matrix (10x10):")
print(confusion_matrix(y_val, val_preds))

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000

Initializing Custom 10-Class SVM (OvR)...
Training all 10 models... (This might take a few minutes in Python)
    Training binary model for Class 0 vs Rest...
    Training binary model for Class 1 vs Rest...
    Training binary model for Class 2 vs Rest...
    Training binary model for Class 3 vs Rest...
    Training binary model for Class 4 vs Rest...
    Training binary model for Class 5 vs Rest...
    Training binary model for Class 6 vs Rest...
    Training binary model for Class 7 vs Rest...
    Training binary model for Class 8 vs Rest...
    Training binary model for Class 9 vs Rest...
Full 10-Class Training completed in 169.91 seconds.

Evaluating on Validation Data...

  10-CLASS MODEL PERFORMANCE (HOG + PCA FEATURES)
              precision    recall  f1-score   support

     Digit 0       0.96      0.98      0.97       587
     Digit 1       0.97      0.98      0.98       630
     Digit 2       0.93

#### CNN


In [2]:
import time
from preprocessing2 import cnn 

X_train_cnn, y_train_cnn, X_val_cnn, y_val_cnn, X_test_cnn, y_test_cnn = cnn(subset_limit=10000)



Extracting CNN Features...
282/282 ━━━━━━━━━━━━━━━━━━━━ 33s 117ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 135ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 45s 143ms/step


#### Cross Validation

In [ ]:

from preprocessing2 import custom_macro_f1_score, k_fold_indices


param_grid = {
    'learning_rate': [0.001, 0.005],
    'lambda_param': [0.01, 0.0001],
    'n_iters': [100, 500]
}

best_f1_score = 0
best_params = {}

folds = k_fold_indices(X_train_cnn, k=3)

total_runs = len(param_grid['learning_rate']) * len(param_grid['lambda_param']) * len(param_grid['n_iters'])
current_run = 1

for lr in param_grid['learning_rate']:
    for lam in param_grid['lambda_param']:
        for iters in param_grid['n_iters']:
            print(f"--- Testing Combo {current_run}/{total_runs} [LR:{lr} | Lam:{lam} | Iters:{iters}] ---")
            fold_f1_scores = []            
            for fold_num, (train_idx, val_idx) in enumerate(folds):
                X_fold_train, y_fold_train = X_train_cnn[train_idx], y_train_cnn[train_idx]
                X_fold_val, y_fold_val = X_train_cnn[val_idx], y_train_cnn[val_idx]
                
                cv_model = OneVsRestSVM(n_classes=10, learning_rate=lr, lambda_param=lam, n_iters=iters)
                cv_model.fit(X_fold_train, y_fold_train)
                
                preds = cv_model.predict(X_fold_val)
                fold_f1 = custom_macro_f1_score(y_fold_val, preds, n_classes=10)
                fold_f1_scores.append(fold_f1)
            
            # Calculate the True Average F1-Score across all 3 folds
            avg_f1 = np.mean(fold_f1_scores)
            print(f"    -> 3-Fold Average Macro F1: {avg_f1:.4f}\n")
            
            # Update the leaderboard based on F1 instead of plain Accuracy
            if avg_f1 > best_f1_score:
                best_f1_score = avg_f1
                best_params = {'learning_rate': lr, 'lambda_param': lam, 'n_iters': iters}
                
            current_run += 1

print("="*50)
print("  CROSS-VALIDATION GRID SEARCH COMPLETE")
print("="*50)
print(f"Highest CV Accuracy: {best_f1_score:.4f}")
print(f"Winning Parameters: {best_params}")

--- Testing Combo 1/8 [LR:0.001 | Lam:0.01 | Iters:100] ---
    Training binary model for Class 0 vs Rest...
    Training binary model for Class 1 vs Rest...
    Training binary model for Class 2 vs Rest...
    Training binary model for Class 3 vs Rest...
    Training binary model for Class 4 vs Rest...
    Training binary model for Class 5 vs Rest...
    Training binary model for Class 6 vs Rest...
    Training binary model for Class 7 vs Rest...
    Training binary model for Class 8 vs Rest...
    Training binary model for Class 9 vs Rest...
    Training binary model for Class 0 vs Rest...
    Training binary model for Class 1 vs Rest...
    Training binary model for Class 2 vs Rest...
    Training binary model for Class 3 vs Rest...
    Training binary model for Class 4 vs Rest...
    Training binary model for Class 5 vs Rest...
    Training binary model for Class 6 vs Rest...
    Training binary model for Class 7 vs Rest...
    Training binary model for Class 8 vs Rest...
    Train

## Testing
#### Best Score Prameters used

In [5]:

from preprocessing2 import custom_classification_report, custom_confusion_matrix

print("Training FINAL Phase 2 model with winning CV parameters...")

final_ovr_cnn_svm = OneVsRestSVM(
    n_classes=10, 
    learning_rate=0.001, 
    lambda_param=0.0001, 
    n_iters=500
)

final_ovr_cnn_svm.fit(X_train_cnn, y_train_cnn)

print("\nRunning final prediction on unseen TEST data...")
final_test_preds_cnn = final_ovr_cnn_svm.predict(X_test_cnn)

print("\n" + "="*50)
print("  OFFICIAL PHASE 2 TEST PERFORMANCE (CNN)")
print("="*50)

target_names = [f"Digit {i}" for i in range(10)]
print(custom_classification_report(y_test_cnn, final_test_preds_cnn, target_names=target_names))

print("\nOfficial Test Confusion Matrix (10x10):")
print(custom_confusion_matrix(y_test_cnn, final_test_preds_cnn, n_classes=10))

Training FINAL Phase 2 model with winning CV parameters...
    Training binary model for Class 0 vs Rest...
    Training binary model for Class 1 vs Rest...
    Training binary model for Class 2 vs Rest...
    Training binary model for Class 3 vs Rest...
    Training binary model for Class 4 vs Rest...
    Training binary model for Class 5 vs Rest...
    Training binary model for Class 6 vs Rest...
    Training binary model for Class 7 vs Rest...
    Training binary model for Class 8 vs Rest...
    Training binary model for Class 9 vs Rest...

Running final prediction on unseen TEST data...

  OFFICIAL PHASE 2 TEST PERFORMANCE (CNN)
                 precision     recall   f1-score    support

Digit 0               0.96       0.97       0.97        980
Digit 1               0.96       0.98       0.97       1135
Digit 2               0.58       0.94       0.72       1032
Digit 3               0.91       0.77       0.83       1010
Digit 4               0.79       0.97       0.87        98